In [1]:
%matplotlib widget

import abtem
import quantem as em
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

import ipywidgets

plt.rcParams['text.color']='white'
plt.rcParams['xtick.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.labelcolor'] = 'white'
plt.rcParams['ytick.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['axes.edgecolor'] = 'white'

plt.rcParams.update({
    "text.usetex": True,
    "text.latex.preamble": r"\usepackage{amsmath}"
})

In [2]:
energy=300e3
gpts=(512, 512)
sampling=(0.1, 0.1)

kwargs = {
    'energy': energy,
    'gpts': gpts,
    'sampling': sampling,
}

def return_ctf(semiangle_cutoff,aberrations=None,**kwargs):
    """ """
    ctf = abtem.CTF(
        semiangle_cutoff=semiangle_cutoff,
        aberration_coefficients=aberrations,
        **kwargs,
    )

    probe = abtem.Probe(
        semiangle_cutoff=semiangle_cutoff,
        aberration_coefficients=aberrations,
        **kwargs,
    )

    fourier_probe = ctf.to_diffraction_patterns(
        gpts=ctf.gpts
    ).array

    return ctf, probe, fourier_probe

ctf_obj, probe_obj, fourier_probe = return_ctf(
    semiangle_cutoff=50,
    aberrations={"C10":-100,"C12":50,"phi12":np.pi/4},
    **kwargs
)

In [3]:
def return_white_obj(energy, gpts, sampling, seed=None):
    """ """
    rng = np.random.default_rng(seed)
    random_phase = rng.normal(scale=np.pi/12,size=gpts)
    sigma = abtem.core.energy.energy2sigma(energy)
    random_pot = random_phase / sigma

    pot = abtem.PotentialArray(
        random_pot[None],
        slice_thickness=0.0,
        sampling=sampling,
    )
    
    return pot

potential_obj = return_white_obj(**kwargs,seed=2026)

In [4]:
scan = [[0,0]]
detector = abtem.PixelatedDetector(max_angle=None)

ronchigram_intensity = probe_obj.multislice(
    potential_obj,
    scan=scan,
    detectors=detector,
    lazy=False,
).array


In [5]:
def array_to_scaled_rgba(array,vmin=0.02,vmax=0.98):
    if np.iscomplexobj(array):
        scaled_amplitude = np.abs(array)
        scaled_angle = np.angle(array)
    else:
        scaled_amplitude = array
        scaled_angle = None

    if scaled_amplitude.std() > 1e-12:
        vmin, vmax = np.quantile(scaled_amplitude,(vmin,vmax))

        scaled_amplitude = (scaled_amplitude.clip(vmin,vmax) - vmin) / (vmax-vmin)
    else:
        scaled_amplitude = np.ones_like(scaled_amplitude)

    rgba = em.visualization.visualization_utils.array_to_rgba(
        scaled_amplitude,
        scaled_angle
    )

    return rgba

In [10]:
width = 620
aspect_ratio = 0.45
height = int(width * aspect_ratio)
dpi = 72

with plt.ioff():
    fig,axs = plt.subplots(1,2,figsize=(width/dpi,height/dpi),dpi=dpi)

rgb_fourier_probe = array_to_scaled_rgba(fourier_probe,vmin=0.001,vmax=0.999)
im_fourier = axs[0].imshow(rgb_fourier_probe)

rgb_ronchigram = array_to_scaled_rgba(ronchigram_intensity,vmin=0.001,vmax=0.999)
im_ronchigram = axs[1].imshow(rgb_ronchigram)

scalebar_fourier = em.visualization.ScalebarConfig(ctf_obj.angular_sampling[0],units='mrad')
titles = ['Fourier-space probe', 'Ronchigram intensity']

for ax, title in zip(axs,titles):
    ax.patch.set_alpha(0)
    ax.set(xticks=[],yticks=[],title=title)

divider = make_axes_locatable(axs[0])
ax_cb = divider.append_axes("right", size="5%", pad="2.5%")
em.visualization.visualization_utils.add_arg_cbar_to_ax(fig,ax_cb)
em.visualization.visualization_utils.add_scalebar_to_ax(
    axs[0],
    rgb_fourier_probe.shape[1],
    scalebar_fourier.sampling,
    scalebar_fourier.length,
    scalebar_fourier.units,
    scalebar_fourier.width_px,
    scalebar_fourier.pad_px,
    scalebar_fourier.color,
    scalebar_fourier.loc,
)

divider = make_axes_locatable(axs[1])
ax_cb = divider.append_axes("right", size="5%", pad="2.5%")

norm_obj = em.core.visualization.CustomNormalization(
    interval_type="quantile",
    lower_quantile=0.001,
    upper_quantile=0.999,
    data=ronchigram_intensity
)

em.visualization.visualization_utils.add_cbar_to_ax(fig,ax_cb,norm_obj,'gray')
em.visualization.visualization_utils.add_scalebar_to_ax(
    axs[1],
    rgb_ronchigram.shape[1],
    scalebar_fourier.sampling,
    scalebar_fourier.length,
    scalebar_fourier.units,
    scalebar_fourier.width_px,
    scalebar_fourier.pad_px,
    scalebar_fourier.color,
    scalebar_fourier.loc,
)

fig.tight_layout()
fig.canvas.resizable = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False
fig.canvas.toolbar_visible = False
fig.canvas.layout.width = f'{width}px'
fig.canvas.toolbar_position = 'bottom'
fig.patch.set_alpha(0)
None

In [16]:
def update_ronchigram(
    semiangle_cutoff,
    defocus_nm,
    astigmatism_nm,
    astigmatism_angle_deg,
    coma_um,
    coma_angle_deg,
):
    """ """
    ctf_obj, probe_obj, fourier_probe = return_ctf(
        semiangle_cutoff=semiangle_cutoff,
        aberrations={
            "C10":-defocus_nm*10,
            "C12":astigmatism_nm*10,
            "phi12":np.deg2rad(astigmatism_angle_deg),
            "C21":coma_um*1e4,
            "phi21":np.deg2rad(coma_angle_deg),
        },
        **kwargs
    )

    ronchigram_intensity = probe_obj.multislice(
        potential_obj,
        scan=scan,
        detectors=detector,
        lazy=False,
    ).array

    rgb_fourier_probe = array_to_scaled_rgba(fourier_probe,vmin=0.001,vmax=0.999)
    im_fourier.set_data(rgb_fourier_probe)

    rgb_ronchigram = array_to_scaled_rgba(ronchigram_intensity,vmin=0.001,vmax=0.999)
    im_ronchigram.set_data(rgb_ronchigram)

    fig.canvas.draw_idle()
    return None

style = {
    'description_width': 'initial',
}

layout = ipywidgets.Layout(width=f'{width//2}px',height='30px')

semiangle_slider = ipywidgets.FloatSlider(
    min=35,
    max=65,
    step=0.5,
    value=50,
    layout=layout,
    style=style,
    description="semi-angle [mrad]",
)

defocus_slider = ipywidgets.FloatSlider(
    min=-15,
    max=15,
    step=0.5,
    value=10,
    layout=layout,
    style=style,
    description="defocus [nm]",
)

astigmatism_slider = ipywidgets.FloatSlider(
    min=0,
    max=10,
    step=0.1,
    value=5,
    layout=layout,
    style=style,
    description="astigmatism [nm]",
)

coma_slider = ipywidgets.FloatSlider(
    min=0,
    max=2,
    step=0.05,
    value=0,
    layout=layout,
    style=style,
    description="coma [μm]",
)

astigmatism_angle_slider = ipywidgets.FloatSlider(
    min=-90,
    max=90,
    step=1,
    value=45,
    layout=layout,
    style=style,
    description="astigmatism angle [°]",
)

coma_angle_slider = ipywidgets.FloatSlider(
    min=-180,
    max=180,
    step=1,
    value=180,
    layout=layout,
    style=style,
    description="coma angle [°]",
)

ipywidgets.interactive_output(
    update_ronchigram,
    {
        'semiangle_cutoff': semiangle_slider,
        'defocus_nm': defocus_slider,
        'astigmatism_nm': astigmatism_slider,
        'astigmatism_angle_deg': astigmatism_angle_slider,
        'coma_um': coma_slider,
        'coma_angle_deg': coma_angle_slider
    }
)
None

In [17]:
#| label: app:ronchigram_widget
display(
    ipywidgets.VBox(
        [
            ipywidgets.HBox([semiangle_slider,defocus_slider]),
            ipywidgets.HBox([astigmatism_slider,astigmatism_angle_slider]),
            ipywidgets.HBox([coma_slider,coma_angle_slider]),
            fig.canvas,
        ],
        layout=ipywidgets.Layout(
            align_items="center"
        )
    )
)